# Week 3 — Data Preprocessing

In [ ]:
import pandas as pd
import numpy as np

# load our filtered dataset
df = pd.read_csv(
"../data/processed/residential_single_family.csv",
parse_dates=["CloseDate"],
low_memory=False
)

df.head()

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,OriginatingSystemName,OriginatingSystemSubName,CloseMonth
0,SanDiego,SanDiego,NaN,True,NaN,NaN,False,3775000.0,1137354161,Sanjay.Solomon@compass.com,...,False,3.0,NaN,92037,NaN,NaN,NaN,NaN,NaN,2025-09
1,TheInlandGateway,TheInlandGateway,NaN,True,NaN,NaN,True,822000.0,1137349439,annb@loislauer.com,...,False,2.0,Redlands Unified,92373,0.0,14560.0,NaN,NaN,NaN,2025-09
2,OrangeCounty,OrangeCounty,"Carpet,Vinyl,Wood",True,NaN,NaN,False,1575000.0,1137346577,Judy@JudyMcCartyRealty.com,...,False,2.0,Saddleback Valley Unified,92630,255.0,3300.0,NaN,NaN,NaN,2025-09
3,BeverlyHillsGreaterLa,BeverlyHillsGreaterLa,"Stone,Tile,Wood",True,NaN,NaN,NaN,8022333.0,1137284778,bba@theagencyre.com,...,False,NaN,NaN,90402,NaN,26981.0,NaN,NaN,NaN,2025-09
4,OrangeCounty,OrangeCounty,NaN,False,NaN,NaN,False,1525000.0,1137269984,ks@kasere.com,...,True,2.0,Huntington Beach Union High,92708,212.0,3749.0,NaN,NaN,NaN,2025-09


##  1. Copy our dataframe and identify the columns

In [32]:
# Create a copy of our dataframe to clean
clean_df = df.copy()

# Numeric columns chosen
numeric_columns = [
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
    "LotSizeSquareFeet", "YearBuilt", "Latitude", "Longitude"
]

# Categorical columns chosen
categorical_columns = ["City", "PostalCode"]

## 2. Clean and format the datatypes

In [33]:
clean_df["CloseDate"] = pd.to_datetime(clean_df["CloseDate"], errors="coerce")

for column in numeric_columns + ["ClosePrice"]:
    values = (clean_df[column].astype("string").str.replace(",", "", regex=False).str.replace("$", "", regex=False))
    clean_df[column] = pd.to_numeric(values, errors="coerce").astype(float)
    clean_df[column] = clean_df[column].replace([np.inf, -np.inf], np.nan)

for column in categorical_columns:
    clean_df[column] = clean_df[column].astype("string").str.strip().replace("", pd.NA)

clean_df["City"] = clean_df["City"].str.upper()
clean_df["PostalCode"] = clean_df["PostalCode"].str.extract(r"^(\d{5})(?:-\d{4}|\.0)?$")[0]

## 3. Handle the invalid records/values

For this, missing values were dropped, and invalid records are changed to NaN (to be later set to our training set's median). Unusable longitude/latitude rows are also dropped.

In [34]:
clean_df = (
    clean_df
    .drop_duplicates()
    .dropna(subset=["ClosePrice", "CloseDate"])
)

clean_df = clean_df.loc[
    clean_df["ClosePrice"] > 0
].copy()

for column in ["LivingArea", "LotSizeSquareFeet", "YearBuilt"]:
    clean_df.loc[
        clean_df[column] <= 0,
        column
    ] = np.nan

clean_df.loc[
    clean_df["YearBuilt"] > clean_df["CloseDate"].dt.year,
    "YearBuilt"
] = np.nan

for column in ["BedroomsTotal", "BathroomsTotalInteger"]:
    invalid = (
        (clean_df[column] < 0)
        | (clean_df[column] % 1 != 0)
    )

    clean_df.loc[invalid, column] = np.nan

# These broad California bounds are a basic sanity check.
clean_df = clean_df.loc[
    clean_df["Latitude"].between(32, 42.1)
    & clean_df["Longitude"].between(-125, -114)
].copy()

## 4. Drop all unwanted columns, display missing summary

In [35]:
keep_columns = (
    ["CloseDate", "ClosePrice"]
    + numeric_columns
    + categorical_columns
)

clean_df = clean_df[keep_columns].copy()

missing_summary = pd.DataFrame({
    "MissingCount": clean_df.isna().sum(),
    "MissingPercent": (
        clean_df.isna().mean() * 100
    ).round(2)
})

display(missing_summary)

,MissingCount,MissingPercent
CloseDate,0,0.00
ClosePrice,0,0.00
LivingArea,73,0.06
BedroomsTotal,0,0.00
BathroomsTotalInteger,13,0.01
LotSizeSquareFeet,2306,1.76
YearBuilt,91,0.07
Latitude,0,0.00
Longitude,0,0.00
City,39,0.03


In [36]:
clean_df.describe()

,CloseDate,ClosePrice,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,YearBuilt,Latitude,Longitude
count,130907,1.309070e+05,130834.000000,130907.000000,130894.000000,1.286010e+05,130816.000000,130907.000000,130907.000000
mean,2025-10-26 00:09:37.509224192,1.334085e+06,2046.536740,3.490738,2.628516,3.907544e+05,1975.779148,34.730010,-118.626383
min,2025-05-01 00:00:00,1.750000e+00,100.000000,0.000000,0.000000,3.000000e-02,1776.000000,32.545249,-124.193201
25%,2025-07-24 00:00:00,6.240000e+05,1386.000000,3.000000,2.000000,5.663000e+03,1956.000000,33.761420,-119.085864
50%,2025-10-20 00:00:00,8.860000e+05,1817.000000,3.000000,2.000000,7.293000e+03,1976.000000,34.082303,-118.025726
75%,2026-02-02 00:00:00,1.409175e+06,2437.000000,4.000000,3.000000,1.045400e+04,1998.000000,34.709384,-117.258662
max,2026-04-30 00:00:00,9.895000e+08,56500.000000,22.000000,35.000000,1.938943e+09,2026.000000,41.894714,-114.347243
std,NaN,8.002582e+06,1042.419351,0.963077,1.127463,1.795044e+07,27.587830,1.691073,1.839326


## 5. Train/Test set
Choose the most recent month of data to be the test set. Keep only the 5th to 95th percentile of data in our training set. Keep X (number of months preceding the test set) as a tunable variable.

In [37]:
X = 11  # Change this to prepare a different training window.

months = clean_df["CloseDate"].dt.to_period("M")
test_month = months.max()

train_df = clean_df.loc[
    (months >= test_month - X)
    & (months < test_month)
].copy()

test_df = clean_df.loc[
    months == test_month
].copy()

if (
    X < 1
    or train_df["CloseDate"].dt.to_period("M").nunique() != X
):
    raise ValueError(
        "Choose a positive X with that many preceding months available."
    )

# Learn price-outlier cutoffs from training data only.
lower_price, upper_price = train_df["ClosePrice"].quantile(
    [0.005, 0.995]
)

train_df = train_df.loc[
    train_df["ClosePrice"].between(lower_price, upper_price)
].copy()

test_df = test_df.loc[
    test_df["ClosePrice"].between(lower_price, upper_price)
].copy()

print("Training months:", test_month - X, "through", test_month - 1)
print("Test month:", test_month)
print("Training rows:", len(train_df), "Test rows:", len(test_df))

Training months: 2025-05 through 2026-03
Test month: 2026-04
Training rows: 117341 Test rows: 12279


## 6. Flag/Impute missing values
Here is where we impute the missing values in our numerical features with the training set's medians. Fill missing room counts with mode (to preserve an integer), and to fill missing categorical features with "Unknown".

In [38]:
# Calculate replacement values using training data only.
fill_values = train_df[numeric_columns].median()

for column in ["BedroomsTotal", "BathroomsTotalInteger"]:
    fill_values[column] = train_df[column].mode().iloc[0]

for data in [train_df, test_df]:

    # Record missingness before filling values.
    for column in numeric_columns + categorical_columns:
        data[column + "_Missing"] = (
            data[column].isna().astype(int)
        )

    data[numeric_columns] = (
        data[numeric_columns].fillna(fill_values)
    )

    data[categorical_columns] = (
        data[categorical_columns].fillna("Unknown")
    )

display(fill_values.rename("TrainingFillValue"))

LivingArea               1815.000000
BedroomsTotal               3.000000
BathroomsTotalInteger       2.000000
LotSizeSquareFeet        7274.000000
YearBuilt                1976.000000
Latitude                   34.081950
Longitude                -118.023632
Name: TrainingFillValue, dtype: float64

## 7. Separate predictors and target + encode categorical columns

In [39]:
X_train = train_df.drop(
    columns=["ClosePrice", "CloseDate"]
).copy()

X_test = test_df.drop(
    columns=["ClosePrice", "CloseDate"]
).copy()

y_train = train_df["ClosePrice"].copy()
y_test = test_df["ClosePrice"].copy()


# Keep only categories with at least this many training examples. Others are replaced with "Other". 
minimum_count = 20

for column in categorical_columns:
    counts = train_df[column].value_counts()

    categories_to_keep = counts[
        counts >= minimum_count
    ].index

    for data in [X_train, X_test]:
        keep = (
            data[column].isin(categories_to_keep)
            | data[column].eq("Unknown")
        )

        data[column] = data[column].where(keep, "Other")

X_train = pd.get_dummies(
    X_train,
    columns=categorical_columns,
    dtype=bool
)

X_test = pd.get_dummies(
    X_test,
    columns=categorical_columns,
    dtype=bool
)

X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=False
)

# Save the prepared datasets to CSV files for later use.
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

## 8. Summary

### Steps completed

- Standardized numerical, categorical, and date formats.
- Checked duplicates and excluded records without a valid closing date
  or positive sale price.
- Converted invalid property measurements to missing values and
  excluded records with unusable coordinates.
- Retained the selected property and location predictors.
- Reserved the most recent month for testing and selected the X months
  before it for training.
- Applied price-outlier thresholds to both datasets.
- Imputed numerical features using training medians and room counts
  using training modes. Missing cities and ZIP codes became `"Unknown"`.
- Added flags indicating which predictor values were missing.
- Grouped cities and ZIP codes with fewer than 20 training records as
  `"Other"` and encoded the categories using one-hot encoding.
- Aligned the training and test predictor columns.
- Separated predictors (`X_train`, `X_test`) from the sale-price targets
  (`y_train`, `y_test`).